In [9]:
import os
from collections import defaultdict
from itertools import chain

import polars as pl

from social_groups.reporting.analysis_columns import AnalysisColumn
from social_groups.reporting.group_reply import (
    GroupReplyAggregator,
    MajorityVote,
    SingularityVote,
)
from social_groups.reporting.parsing import (
    AnswerOptions,
    AnswerParser,
    AnswerComparer,
)
from social_groups.directories import REPORTING_DIR
from social_groups.reporting.plots.config import MODEL_NAME_TO_LETTER_MAPPING
from social_groups.reporting.group_decision_scheme import calculate_decision_scheme

from social_groups.reporting.heterogenous_comparison.data_retrieval import (
    get_baseline_frame,
    get_mad_frame,
    apply_parsing_and_group_decision,
)

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [10]:
output_dir = REPORTING_DIR / "heterogeneous_group"
os.makedirs(output_dir, exist_ok=True)

triple_underscore_handling = "wrong"

In [11]:
parser = AnswerParser(AnswerOptions.letters_A_to_J)
group_reply = GroupReplyAggregator(MajorityVote())
comparer = AnswerComparer(
    AnswerOptions.letters_A_to_J, triple_underscore_handling=triple_underscore_handling
)

In [12]:
baseline_frame = get_baseline_frame()

In [13]:
baseline_frame

id,run_id,question_id,phoenix_span_id,phoenix_span_url,run_identifier,final_answer,original_question_id,category,question,answer_string,model_name
i64,i64,i64,str,str,str,str,i64,str,str,str,str
0,0,0,"""fc4fba7e31996dbd""","""http://localhost:6006/projects…","""2026-01-28-11-18-38 - heteroge…","""<think> Okay, let's tackle thi…",9456,"""physics""","""Q: An electric dipole consisti…","""I""","""Qwen/Qwen3-4B"""
1,0,1,"""a07b8366a728060b""","""http://localhost:6006/projects…","""2026-01-28-11-18-38 - heteroge…","""<think> Okay, let's try to fig…",5681,"""other""","""Q: In 2018, about how many chi…","""G""","""Qwen/Qwen3-4B"""
2,0,2,"""c9ce50b76ffe168f""","""http://localhost:6006/projects…","""2026-01-28-11-18-38 - heteroge…","""<think> Okay, so the question …",6622,"""health""","""Q: What stable isotope is comm…","""J""","""Qwen/Qwen3-4B"""
3,0,3,"""87b17edffe475e00""","""http://localhost:6006/projects…","""2026-01-28-11-18-38 - heteroge…","""<think> Okay, let's try to fig…",1190,"""law""","""Q: A company contracted with a…","""E""","""Qwen/Qwen3-4B"""
4,0,4,"""0504d6d4cbbfb41c""","""http://localhost:6006/projects…","""2026-01-28-11-18-38 - heteroge…","""<think> Okay, let's tackle thi…",4231,"""chemistry""","""Q: A sample of bristle cone pi…","""E""","""Qwen/Qwen3-4B"""
…,…,…,…,…,…,…,…,…,…,…,…
311,2,105,"""30f58a6b4f8f34f4""","""http://localhost:6006/projects…","""2026-01-28-11-18-38 - heteroge…","""<think> Okay, let's try to fig…",396,"""business""","""Q: Mrs. Reynolds purchased $45…","""J""","""Qwen/Qwen3-14B"""
350,2,112,"""c15ac7e81840a074""","""http://localhost:6006/projects…","""2026-01-28-11-18-38 - heteroge…","""<think> Okay, let's try to fig…",7140,"""economics""","""Q: In a given economy, househo…","""A""","""Qwen/Qwen3-14B"""
351,2,152,"""92edc94fb7524a01""","""http://localhost:6006/projects…","""2026-01-28-11-18-38 - heteroge…","""<think> Okay, let's try to fig…",8341,"""math""","""Q: In the jury pool available …","""B""","""Qwen/Qwen3-14B"""


In [14]:
print("Number of unparsable answers:")
print(
    baseline_frame.with_columns(
        parser(pl.col("final_answer")).alias(AnalysisColumn.parsed_answer.value)
    )
    .group_by("model_name")
    .agg(
        no_null=pl.col(AnalysisColumn.parsed_answer.value)
        .str.starts_with("___")
        .not_()
        .sum(),
        null_percentage=(
            pl.col(AnalysisColumn.parsed_answer.value).str.starts_with("___").mean()
            * 100
        ).round(2),
    )
    .sort(pl.col("model_name").str.extract(r"-(\d+\.?\d*)B", 1).cast(pl.Float64))
)

baseline_analysis = (
    baseline_frame.with_columns(
        parser(pl.col("final_answer")).alias(AnalysisColumn.parsed_answer.value)
    )
    .with_columns(
        is_correct=comparer(
            pl.col(AnalysisColumn.parsed_answer.value), pl.col("answer_string")
        )
    )
    .group_by("model_name")
    .agg(accuracy=pl.col("is_correct").mean())
    .sort(pl.col("model_name").str.extract(r"-(\d+\.?\d*)B", 1).cast(pl.Float64))
)

baseline_analysis

Number of unparsable answers:
shape: (3, 3)
┌─────────────────┬─────────┬─────────────────┐
│ model_name      ┆ no_null ┆ null_percentage │
│ ---             ┆ ---     ┆ ---             │
│ str             ┆ u32     ┆ f64             │
╞═════════════════╪═════════╪═════════════════╡
│ Qwen/Qwen3-0.6B ┆ 90      ┆ 10.0            │
│ Qwen/Qwen3-4B   ┆ 88      ┆ 12.0            │
│ Qwen/Qwen3-14B  ┆ 93      ┆ 7.0             │
└─────────────────┴─────────┴─────────────────┘


model_name,accuracy
str,f64
"""Qwen/Qwen3-0.6B""",0.35
"""Qwen/Qwen3-4B""",0.59
"""Qwen/Qwen3-14B""",0.64


## Analzying Basic Group Behaviour

In [15]:
mad_frame = get_mad_frame()
mad_frame

shape: (3_400, 17)
┌──────┬────────┬─────────────┬────────────┬───┬────────────┬────────────┬────────────┬────────────┐
│ id   ┆ run_id ┆ question_id ┆ phoenix_sp ┆ … ┆ category   ┆ question   ┆ answer_str ┆ model_name │
│ ---  ┆ ---    ┆ ---         ┆ an_id      ┆   ┆ ---        ┆ ---        ┆ ing        ┆ s          │
│ i64  ┆ i64    ┆ i64         ┆ ---        ┆   ┆ str        ┆ str        ┆ ---        ┆ ---        │
│      ┆        ┆             ┆ str        ┆   ┆            ┆            ┆ str        ┆ list[str]  │
╞══════╪════════╪═════════════╪════════════╪═══╪════════════╪════════════╪════════════╪════════════╡
│ 294  ┆ 3      ┆ 45          ┆ ee9c12870a ┆ … ┆ physics    ┆ Q:         ┆ F          ┆ ["Qwen/Qwe │
│      ┆        ┆             ┆ 3b979e     ┆   ┆            ┆ Kirkwood   ┆            ┆ n3-14B"]   │
│      ┆        ┆             ┆            ┆   ┆            ┆ gaps are   ┆            ┆            │
│      ┆        ┆             ┆            ┆   ┆            ┆ observed … ┆            ┆            │
│ 295  ┆ 3      ┆ 79          ┆ 1d6ae1faca ┆ … ┆ health     ┆ Q: A 2-mon ┆ F          ┆ ["Qwen/Qwe │
│      ┆        ┆             ┆ cf8002     ┆   ┆            ┆ th-old     ┆            ┆ n3-14B"]   │
│      ┆        ┆             ┆            ┆   ┆            ┆ female is  ┆            ┆            │
│      ┆        ┆             ┆            ┆   ┆            ┆ bro…       ┆            ┆            │
│ 296  ┆ 3      ┆ 95          ┆ f9d891b522 ┆ … ┆ philosophy ┆ Q:  In     ┆ D          ┆ ["Qwen/Qwe │
│      ┆        ┆             ┆ e3eec7     ┆   ┆            ┆ response   ┆            ┆ n3-14B"]   │
│      ┆        ┆             ┆            ┆   ┆            ┆ to the     ┆            ┆            │
│      ┆        ┆             ┆            ┆   ┆            ┆ argumen…   ┆            ┆            │
│ 297  ┆ 3      ┆ 22          ┆ 0bdce6238f ┆ … ┆ biology    ┆ Q: What    ┆ F          ┆ ["Qwen/Qwe │
│      ┆        ┆             ┆ 1ca68f     ┆   ┆            ┆ roles does ┆            ┆ n3-14B"]   │
│      ┆        ┆             ┆            ┆   ┆            ┆ glucose    ┆            ┆            │
│      ┆        ┆             ┆            ┆   ┆            ┆ pla…       ┆            ┆            │
│ 298  ┆ 3      ┆ 61          ┆ 607c013c34 ┆ … ┆ health     ┆ Q: Blood   ┆ H          ┆ ["Qwen/Qwe │
│      ┆        ┆             ┆ 516755     ┆   ┆            ┆ clots are  ┆            ┆ n3-14B"]   │
│      ┆        ┆             ┆            ┆   ┆            ┆ responsibl ┆            ┆            │
│      ┆        ┆             ┆            ┆   ┆            ┆ e…         ┆            ┆            │
│ …    ┆ …      ┆ …           ┆ …          ┆ … ┆ …          ┆ …          ┆ …          ┆ …          │
│ 3695 ┆ 36     ┆ 52          ┆ 8599cb5bf0 ┆ … ┆ biology    ┆ Q: What    ┆ D          ┆ ["Qwen/Qwe │
│      ┆        ┆             ┆ c91c9d     ┆   ┆            ┆ are the    ┆            ┆ n3-0.6B",  │
│      ┆        ┆             ┆            ┆   ┆            ┆ difficulti ┆            ┆ "Qwen/Qwen │
│      ┆        ┆             ┆            ┆   ┆            ┆ es i…      ┆            ┆ …          │
│ 3696 ┆ 36     ┆ 5           ┆ b113d185f8 ┆ … ┆ philosophy ┆ Q:  When   ┆ A          ┆ ["Qwen/Qwe │
│      ┆        ┆             ┆ 688d64     ┆   ┆            ┆ was the    ┆            ┆ n3-0.6B",  │
│      ┆        ┆             ┆            ┆   ┆            ┆ major      ┆            ┆ "Qwen/Qwen │
│      ┆        ┆             ┆            ┆   ┆            ┆ shift b…   ┆            ┆ …          │
│ 3697 ┆ 36     ┆ 9           ┆ a82a93c4d7 ┆ … ┆ biology    ┆ Q: What is ┆ I          ┆ ["Qwen/Qwe │
│      ┆        ┆             ┆ a8be41     ┆   ┆            ┆ differenti ┆            ┆ n3-0.6B",  │
│      ┆        ┆             ┆            ┆   ┆            ┆ al reprod… ┆            ┆ "Qwen/Qwen │
│      ┆        ┆             ┆            ┆   ┆            ┆            ┆            ┆ …          │
│ 3698 ┆ 36     ┆ 83          ┆ 3f971bfed7

In [16]:
model_name_sort = {
    "Qwen/Qwen3-14B": 1,
    "Qwen/Qwen3-4B": 2,
    "Qwen/Qwen3-0.6B": 3,
}

mad_analysis = apply_parsing_and_group_decision(
    mad_frame, parser, comparer, group_reply
)

table_page_46 = pl.concat(
    [
        (
            mad_analysis.group_by("group_constellation")
            .agg(accuracy=pl.col("is_correct").mean())
            .with_columns(origin=pl.lit("(MAD)"))
        ),
        (
            baseline_analysis.with_columns(
                group_constellation=pl.col("model_name").replace(
                    MODEL_NAME_TO_LETTER_MAPPING
                ),
                origin=pl.lit("(baseline)"),
            ).drop("model_name")
        ),
    ],
    how="diagonal",
)

table_page_46.write_csv(output_dir / "llm_based_table_page_46.csv")

table_page_46

group_constellation,accuracy,origin
str,f64,str
"""HL""",0.54,"""(MAD)"""
"""LM""",0.52,"""(MAD)"""
"""LL""",0.31,"""(MAD)"""
"""MMMM""",0.59,"""(MAD)"""
"""LLLL""",0.28,"""(MAD)"""
…,…,…
"""HHLM""",0.69,"""(MAD)"""
"""HHHH""",0.69,"""(MAD)"""
"""L""",0.35,"""(baseline)"""


In [17]:
original_table_page46 = pl.DataFrame(
    {
        "group_constellation": [
            "HHH",
            "HHM",
            "HHL",
            "HML",
            "HMM",
            "H",
            "HLL",
            "MMM",
            "M",
            "MML",
            "MLL",
            "L",
            "LLL",
        ],
        "score (-115 to 115)": [80, 74, 67, 64, 61, 60, 56, 48, 42, 39, 37, 25, 21],
    }
)
original_table_page46.write_csv(output_dir / "human_based_table_page_46.csv")
original_table_page46

group_constellation,score (-115 to 115)
str,i64
"""HHH""",80
"""HHM""",74
"""HHL""",67
"""HML""",64
"""HMM""",61
…,…
"""M""",42
"""MML""",39
"""MLL""",37


In [18]:
decision_schemes = (
    mad_analysis.group_by("group_constellation")
    .map_groups(
        lambda g: calculate_decision_scheme(
            g,
            AnalysisColumn.parsed_individual_answers_before.value,
            AnalysisColumn.parsed_individual_answers_after.value,
            "answer_string",
            group_reply,
            comparer,
        ).select(
            pl.lit(g["group_constellation"].unique().item()).alias(
                "group_constellation"
            ),
            "Correct Members Beginning",
            "correct",
            "incorrect",
        )
    )
    .sort("group_constellation")
)

decision_schemes.write_csv(output_dir / "group_decision_schemes.csv")

decision_schemes

group_constellation,Correct Members Beginning,correct,incorrect
str,u32,f64,f64
"""H""",1,0.944444,0.055556
"""H""",0,0.142857,0.857143
"""HH""",2,0.983871,0.016129
"""HH""",1,0.6875,0.3125
"""HH""",0,0.181818,0.818182
…,…,…,…
"""MMMM""",4,0.925926,0.074074
"""MMMM""",3,0.666667,0.333333
"""MMMM""",2,0.666667,0.333333


## Unparsable answers:

In [19]:
unparsable_answers_per_model = defaultdict(int)

### In the baseline:

In [20]:
for answer in (
    baseline_frame.with_columns(
        parser(pl.col("final_answer")).alias(AnalysisColumn.parsed_answer.value)
    )
    .filter(pl.col(AnalysisColumn.parsed_answer.value).str.starts_with("___"))
    .select(["final_answer", "model_name"])
    .iter_rows()
):
    print(answer[1], ": \n")
    print(answer[0][-150:])
    print("-" * 50)
    unparsable_answers_per_model[answer[1]] += 1

Qwen/Qwen3-0.6B : 

he other options don't address the government's right to protect the informant's identity. So the answer should be G.
</think>

The answer is ... (G).
--------------------------------------------------
Qwen/Qwen3-0.6B : 

on again. The two times are 6:12:43 and 6:52:43. The difference is 40 minutes. So if he left at 6:12:43, he was away for 40 minutes. But since he left
--------------------------------------------------
Qwen/Qwen3-4B : 

sec². So, if I have h1 - h2 in Btu/lbm, I need to convert it to ft²/sec². Let me use the following conversion:

1 Btu/lbm = 778.169 ft-lbf/lbm

Then, 
--------------------------------------------------
Qwen/Qwen3-4B : 

. 

Alternatively, maybe the problem is using the value of Cv for CO2 as 25.1 J/mol·K. 

Alternatively, maybe the problem is using the value of Cv for
--------------------------------------------------
Qwen/Qwen3-0.6B : 

ybe there's a mistake in the options? Or perhaps I misread the problem. 

Wait, looking back at 

## In the MAD:

In [21]:
for answer in chain(
    mad_analysis.filter(
        pl.col(AnalysisColumn.parsed_individual_answers_before.value)
        .list.eval(pl.element().str.starts_with("___"))
        .list.any()
    )
    .select(
        "answers_at_beginning",
        AnalysisColumn.parsed_individual_answers_before.value,
        "model_names",
    )
    .iter_rows(),
    mad_analysis.filter(
        pl.col(AnalysisColumn.parsed_individual_answers_before.value)
        .list.eval(pl.element().str.starts_with("___"))
        .list.any()
    )
    .select(
        "answers_at_end",
        AnalysisColumn.parsed_individual_answers_after.value,
        "model_names",
    )
    .iter_rows(),
):
    for i, parsed in enumerate(answer[1]):
        if parsed.startswith("___"):
            unparsable_answers_per_model[answer[2][i]] += 1
            print(parsed, f"from {answer[2][i]}: \n")
            print(answer[0][i][-150:])
            print("-" * 50)

___not_parsable___ from Qwen/Qwen3-14B: 

t budget deficit plus the difference in tax revenues and government spending due to the change in GNP. 

But how? 

Alternatively, the full employment
--------------------------------------------------
___not_parsable___ from Qwen/Qwen3-14B: 

 Converting to ft²/sec², since 1 ft·lbf = 32.174 ft²/sec², so R = 85.78 * 32.174 ≈ 2760 ft²/sec²·°R.

Temperature T1 is 600°F, which is 600 + 459.67 =
--------------------------------------------------
___not_parsable___ from Qwen/Qwen3-14B: 

ħ^2) / (μ_0 * 4π * r^3) * something. But without knowing r, this is not helpful.

Alternatively, the hyperfine splitting in terms of magnetic field is
--------------------------------------------------
___not_parsable___ from Qwen/Qwen3-14B: 

 planes. Then d = a / sqrt(2) ≈ 6.29 / 1.414 ≈ 4.45 Å. Then:

sinθ = 0.248 / (2 * 4.45) ≈ 0.248 / 8.9 ≈ 0.0279

θ ≈ 1.6 degrees. Still not matching.


--------------------------------------------------
___not_parsable___ from 

-> Qwen 0.6B often says "Answer should be A /think. The Answer is G"

In [22]:
unparsable_answers_per_model

defaultdict(int,
            {'Qwen/Qwen3-0.6B': 856,
             'Qwen/Qwen3-4B': 376,
             'Qwen/Qwen3-14B': 317})

### Build an overview of parsing error Influence

In [23]:
baseline_frame_with_group_constellation = baseline_frame.with_columns(
    group_constellation=pl.col("model_name").replace(MODEL_NAME_TO_LETTER_MAPPING)
    + pl.lit(" (Baseline)")
).drop("model_name")

parsing_error_influence_table = pl.DataFrame(
    {
        "group_constellation": mad_analysis["group_constellation"]
        .unique()
        .extend(baseline_frame_with_group_constellation["group_constellation"].unique())
        .sort()
    }
)

for handling in ["null", "wrong", "random"]:
    new_comparer = AnswerComparer(AnswerOptions.letters_A_to_J, handling)

    new_mad = (
        apply_parsing_and_group_decision(mad_frame, parser, new_comparer, group_reply)
        .group_by("group_constellation")
        .agg(accuracy=pl.col("is_correct").mean())
    )

    new_base = (
        baseline_frame_with_group_constellation.with_columns(
            parser(pl.col("final_answer")).alias(AnalysisColumn.parsed_answer.value)
        )
        .with_columns(
            is_correct=new_comparer(
                pl.col(AnalysisColumn.parsed_answer.value), pl.col("answer_string")
            )
        )
        .group_by("group_constellation")
        .agg(accuracy=pl.col("is_correct").mean())
    )

    parsing_error_influence_table = parsing_error_influence_table.join(
        pl.concat([new_mad, new_base], how="diagonal").select(
            "group_constellation",
            pl.col("accuracy").alias(f"Accuracy ({handling})"),
        ),
        on="group_constellation",
        how="inner",
    )

parsing_error_influence_table.with_columns(
    deviation=(
        pl.max_horizontal(pl.exclude("group_constellation"))
        - pl.min_horizontal(pl.exclude("group_constellation"))
    ).alias("range")
)

group_constellation,Accuracy (null),Accuracy (wrong),Accuracy (random),deviation
str,f64,f64,f64,f64
"""HHL""",0.663265,0.65,0.65,0.013265
"""HHM""",0.714286,0.7,0.7,0.014286
"""HLMM""",0.622449,0.61,0.62,0.012449
"""HHLM""",0.71875,0.69,0.69,0.02875
"""LLMM""",0.622222,0.56,0.56,0.062222
…,…,…,…,…
"""HM""",0.685393,0.61,0.64,0.075393
"""HHHM""",0.659794,0.64,0.64,0.019794
"""M (Baseline)""",0.670455,0.59,0.59,0.080455


-> Using (null) is the best, as unparsed values increase your score... should not be used

-> Using wrong is the worst, could possibly be used to be fair, as it is "not correct"

-> Papers and Benchmarks often use "random", which increases the values artificially and introduces noise.. I do not like it, but to be fair one should use it.

---

-> But in all of out cases, even absolute deviation is actually pretty low. (it gets mitigated in group decisions, as unparsable values are ignored in aggregation)

# Agreeableness -> Number of Cases where they reach conslusion

In [24]:
unanimity = (
    mad_analysis.with_columns(
        pl.col(AnalysisColumn.parsed_individual_answers_after.value)
        .list.n_unique()
        .eq(1)
        .alias("End unanimity"),
        pl.col(AnalysisColumn.parsed_individual_answers_before.value)
        .list.n_unique()
        .eq(1)
        .alias("Start unanimity"),
    )
    .group_by("group_constellation")
    .agg(
        pl.col("End unanimity").mean(),
        pl.col("Start unanimity").mean(),
        pl.when(pl.col("Start unanimity"))
        .then(None)
        .otherwise(pl.col("End unanimity"))
        .mean()
        .alias("End Unanimity | not Start Unanimity"),
        pl.when(pl.col("Start unanimity"))
        .then(pl.col("End unanimity"))
        .otherwise(None)
        .mean()
        .alias("End Unanimity | Start Unanimity"),
        accuracy=pl.col("is_correct").mean(),
    )
)

unanimity

group_constellation,End unanimity,Start unanimity,End Unanimity | not Start Unanimity,End Unanimity | Start Unanimity,accuracy
str,f64,f64,f64,f64,f64
"""HLM""",0.85,0.25,0.813333,0.96,0.67
"""HHLM""",0.83,0.29,0.802817,0.896552,0.69
"""HHH""",0.95,0.75,0.84,0.986667,0.73
"""HLL""",0.55,0.22,0.461538,0.863636,0.47
"""HHHL""",0.71,0.29,0.647887,0.862069,0.64
…,…,…,…,…,…
"""HMMM""",0.91,0.72,0.928571,0.902778,0.65
"""HM""",0.88,0.74,0.846154,0.891892,0.61
"""HLMM""",0.79,0.2,0.775,0.85,0.61


-> In Human Groups (see Group Problem Solving) there is the tendency that the smarter the group, the more it is a "Truth Supported" Decision Scheme, the "dumber" the group, the more it is "proportional"

### Correlation for groups (ignoring 1 member, as it is always 1)

In [25]:
print("Pearson Correlation:")
print(
    unanimity.filter(pl.col("group_constellation").str.len_chars() > 1)
    .select(pl.exclude("group_constellation"))
    .corr()
)

print("Spearman Rank Correlation:")

print(
    unanimity.filter(pl.col("group_constellation").str.len_chars() > 1)
    .select(pl.exclude("group_constellation"))
    .with_columns(pl.all().rank())
    .corr()
)

Pearson Correlation:
shape: (5, 5)
┌───────────────┬─────────────────┬─────────────────────┬───────────────────────┬──────────┐
│ End unanimity ┆ Start unanimity ┆ End Unanimity | not ┆ End Unanimity | Start ┆ accuracy │
│ ---           ┆ ---             ┆ Start Unan…         ┆ Unanimit…             ┆ ---      │
│ f64           ┆ f64             ┆ ---                 ┆ ---                   ┆ f64      │
│               ┆                 ┆ f64                 ┆ f64                   ┆          │
╞═══════════════╪═════════════════╪═════════════════════╪═══════════════════════╪══════════╡
│ 1.0           ┆ 0.827728        ┆ 0.927171            ┆ 0.527695              ┆ 0.420484 │
│ 0.827728      ┆ 1.0             ┆ 0.716343            ┆ 0.476741              ┆ 0.335886 │
│ 0.927171      ┆ 0.716343        ┆ 1.0                 ┆ 0.252307              ┆ 0.416728 │
│ 0.527695      ┆ 0.476741        ┆ 0.252307            ┆ 1.0                   ┆ 0.191466 │
│ 0.420484      ┆ 0.335886        ┆

-> significantly correlated

---
-> Accuracy Correlates with Unanimity
Of course this could be that if everyone is correct in the beginning, then if they that way, then the chance of being correct is higher,
But can we increase the accuracy by accepting when a group starts with one answer?

## Analysis: Is the group finding answers "together" ? (e.g. can it find answers out of wrong start)

In [26]:
correctness_influence = (
    mad_analysis.with_columns(
        correct_before=comparer(
            pl.col(AnalysisColumn.parsed_combined_answers_before.value),
            pl.col("answer_string"),
        )
    )
    .group_by("group_constellation")
    .agg(
        pl.when(pl.col("correct_before"))
        .then(pl.col("is_correct"))
        .otherwise(None)
        .mean()
        .alias("Correct | Correct in Beginning"),
        pl.when(pl.col("correct_before"))
        .then(None)
        .otherwise(pl.col("is_correct"))
        .mean()
        .alias("Correct | !Correct in Beginning"),
        accuracy=pl.col("is_correct").mean(),
    )
)

correctness_influence

group_constellation,Correct | Correct in Beginning,Correct | !Correct in Beginning,accuracy
str,f64,f64,f64
"""MMM""",0.9,0.1,0.58
"""LLM""",0.860465,0.333333,0.56
"""LM""",0.885714,0.323077,0.52
"""HHLL""",0.803922,0.204082,0.51
"""MMMM""",0.901639,0.102564,0.59
…,…,…,…
"""HLL""",0.880952,0.172414,0.47
"""HLLL""",0.813953,0.140351,0.43
"""HHL""",0.923077,0.142857,0.65


While the difference in Correct | Correct in Beginning is negligible, the true power lies in changing the answer when They are not correct.

In [27]:
(
    correctness_influence.join(unanimity, how="left", on="group_constellation")
    .filter(pl.col("group_constellation").str.len_chars() > 1)
    .select(pl.exclude("group_constellation"))
    .with_columns(pl.all().rank())
    .corr()
)

Correct | Correct in Beginning,Correct | !Correct in Beginning,accuracy,End unanimity,Start unanimity,End Unanimity | not Start Unanimity,End Unanimity | Start Unanimity,accuracy_right
f64,f64,f64,f64,f64,f64,f64,f64
1.0,0.01958,0.714517,0.472904,0.351131,0.463742,0.419853,0.714517
0.01958,1.0,-0.12255,-0.351177,-0.391212,-0.245079,-0.149016,-0.12255
0.714517,-0.12255,1.0,0.493686,0.349566,0.502777,0.285166,1.0
0.472904,-0.351177,0.493686,1.0,0.8586,0.878381,0.618211,0.493686
0.351131,-0.391212,0.349566,0.8586,1.0,0.754214,0.455629,0.349566
0.463742,-0.245079,0.502777,0.878381,0.754214,1.0,0.297922,0.502777
0.419853,-0.149016,0.285166,0.618211,0.455629,0.297922,1.0,0.285166
0.714517,-0.12255,1.0,0.493686,0.349566,0.502777,0.285166,1.0


## Influence of Group Aggregation Protocol

In [28]:
group_reply_influence_table = pl.DataFrame(
    {
        "group_constellation": mad_analysis["group_constellation"]
        .unique()
        .extend(baseline_frame_with_group_constellation["group_constellation"].unique())
        .sort()
    }
)

for strategy in [MajorityVote(), SingularityVote()]:
    new_group_reply_agg = GroupReplyAggregator(strategy)
    group_reply_influence_table = group_reply_influence_table.join(
        apply_parsing_and_group_decision(
            mad_frame, parser, comparer, new_group_reply_agg
        )
        .group_by("group_constellation")
        .agg(accuracy=pl.col("is_correct").mean())
        .select(
            "group_constellation",
            pl.col("accuracy").alias(f"Accuracy ({strategy.__class__.__name__})"),
        ),
        on="group_constellation",
        how="inner",
    )

group_reply_influence_table = group_reply_influence_table.with_columns(
    deviation=(
        pl.max_horizontal(pl.exclude("group_constellation"))
        - pl.min_horizontal(pl.exclude("group_constellation"))
    ).alias("range")
)

group_reply_influence_table

group_constellation,Accuracy (MajorityVote),Accuracy (SingularityVote),deviation
str,f64,f64,f64
"""HL""",0.54,0.51,0.03
"""MM""",0.61,0.6,0.01
"""HLLM""",0.58,0.54,0.04
"""HHH""",0.73,0.7,0.03
"""L""",0.25,0.25,0.0
…,…,…,…
"""LL""",0.31,0.31,0.0
"""HHHL""",0.64,0.54,0.1
"""HHM""",0.7,0.64,0.06


In [29]:
group_reply_influence_table.drop("group_constellation").with_columns(
    pl.all().rank()
).corr()

Accuracy (MajorityVote),Accuracy (SingularityVote),deviation
f64,f64,f64
1.0,0.93389,0.050611
0.93389,1.0,-0.225041
0.050611,-0.225041,1.0


-> Slightly Negative Correlation between Accuracy and the deviation, meaning the better the more MajorityVote == SingularityVote -> Same argument as before

## Inter-Model Correctness Correlation (Aka answer diversity)

In [30]:
mad_analysis

shape: (3_400, 23)
┌──────┬────────┬─────────────┬────────────┬───┬────────────┬────────────┬────────────┬────────────┐
│ id   ┆ run_id ┆ question_id ┆ phoenix_sp ┆ … ┆ group_cons ┆ ___parsed_ ┆ ___parsed_ ┆ is_correct │
│ ---  ┆ ---    ┆ ---         ┆ an_id      ┆   ┆ tellation  ┆ combined_a ┆ combined_a ┆ ---        │
│ i64  ┆ i64    ┆ i64         ┆ ---        ┆   ┆ ---        ┆ nswers_bef ┆ nswers_aft ┆ bool       │
│      ┆        ┆             ┆ str        ┆   ┆ str        ┆ …          ┆ …          ┆            │
│      ┆        ┆             ┆            ┆   ┆            ┆ ---        ┆ ---        ┆            │
│      ┆        ┆             ┆            ┆   ┆            ┆ str        ┆ str        ┆            │
╞══════╪════════╪═════════════╪════════════╪═══╪════════════╪════════════╪════════════╪════════════╡
│ 294  ┆ 3      ┆ 45          ┆ ee9c12870a ┆ … ┆ H          ┆ F          ┆ F          ┆ true       │
│      ┆        ┆             ┆ 3b979e     ┆   ┆            ┆            ┆            ┆            │
│ 295  ┆ 3      ┆ 79          ┆ 1d6ae1faca ┆ … ┆ H          ┆ F          ┆ F          ┆ true       │
│      ┆        ┆             ┆ cf8002     ┆   ┆            ┆            ┆            ┆            │
│ 296  ┆ 3      ┆ 95          ┆ f9d891b522 ┆ … ┆ H          ┆ C          ┆ C          ┆ false      │
│      ┆        ┆             ┆ e3eec7     ┆   ┆            ┆            ┆            ┆            │
│ 297  ┆ 3      ┆ 22          ┆ 0bdce6238f ┆ … ┆ H          ┆ F          ┆ F          ┆ true       │
│      ┆        ┆             ┆ 1ca68f     ┆   ┆            ┆            ┆            ┆            │
│ 298  ┆ 3      ┆ 61          ┆ 607c013c34 ┆ … ┆ H          ┆ H          ┆ H          ┆ true       │
│      ┆        ┆             ┆ 516755     ┆   ┆            ┆            ┆            ┆            │
│ …    ┆ …      ┆ …           ┆ …          ┆ … ┆ …          ┆ …          ┆ …          ┆ …          │
│ 3695 ┆ 36     ┆ 52          ┆ 8599cb5bf0 ┆ … ┆ LMM        ┆ D          ┆ D          ┆ true       │
│      ┆        ┆             ┆ c91c9d     ┆   ┆            ┆            ┆            ┆            │
│ 3696 ┆ 36     ┆ 5           ┆ b113d185f8 ┆ … ┆ LMM        ┆ C          ┆ C          ┆ false      │
│      ┆        ┆             ┆ 688d64     ┆   ┆            ┆            ┆            ┆            │
│ 3697 ┆ 36     ┆ 9           ┆ a82a93c4d7 ┆ … ┆ LMM        ┆ I          ┆ I          ┆ true       │
│      ┆        ┆             ┆ a8be41     ┆   ┆            ┆            ┆            ┆            │
│ 3698 ┆ 36     ┆ 83          ┆ 3f971bfed7 ┆ … ┆ LMM        ┆ C          ┆ C          ┆ true       │
│      ┆        ┆             ┆ 2f5e1c     ┆   ┆            ┆            ┆            ┆            │
│ 3699 ┆ 36     ┆ 90          ┆ e0ea217990 ┆ … ┆ LMM        ┆ D          ┆ D          ┆ true       │
│      ┆        ┆             ┆ 89f5d4     ┆   ┆            ┆            ┆            ┆            │
└──────┴────────┴─────────────┴────────────┴───┴────────────┴────────────┴────────────┴────────────┘

#### For Individual answers

In [31]:
individual_models_answer_per_question = (
    (
        baseline_frame.with_columns(
            parser(pl.col("final_answer")).alias(AnalysisColumn.parsed_answer.value)
        )
        .with_columns(
            pl.col("model_name").replace(MODEL_NAME_TO_LETTER_MAPPING),
            is_correct=comparer(
                pl.col(AnalysisColumn.parsed_answer.value), pl.col("answer_string")
            ),
        )
        .pivot(
            values="is_correct",
            index="question_id",
            on="model_name",
            aggregate_function="mean",
        )
    )
    .sort("question_id")
    .select("question_id", "L", "M", "H")
    .with_columns(pl.exclude("question_id").cast(bool))
)

individual_models_answer_per_question

question_id,L,M,H
i64,bool,bool,bool
0,true,true,true
1,false,true,true
2,true,true,true
3,true,false,false
4,false,true,false
…,…,…,…
152,true,false,true
153,true,true,true
154,false,true,true


In [32]:
individual_models_answer_per_question.drop("question_id").corr()

L,M,H
f64,f64,f64
1.0,0.313313,0.2446
0.313313,1.0,0.603185
0.2446,0.603185,1.0


--> Actually surprisingly different

In [33]:
df = pl.concat(
    [
        individual_models_answer_per_question.group_by(col)
        .agg(pl.all().exclude(col, "question_id").mean())
        .with_columns(
            pl.when(pl.col(col))
            .then(pl.lit(col + "_correct"))
            .otherwise(pl.lit(col + "_incorrect"))
            .alias("Given"),
            pl.when(pl.col(col)).then(pl.lit(1.0)).otherwise(pl.lit(0.0)).alias(col),
        )
        for col in individual_models_answer_per_question.columns
        if col != "question_id"
    ],
    how="diagonal",
)

df_single = df.select("Given", "L", "M", "H")

df_single

Given,L,M,H
str,f64,f64,f64
"""L_correct""",1.0,0.8,0.8
"""L_incorrect""",0.0,0.476923,0.553846
"""M_incorrect""",0.170732,0.0,0.292683
"""M_correct""",0.474576,1.0,0.881356
"""H_correct""",0.4375,0.8125,1.0
"""H_incorrect""",0.194444,0.194444,0.0


-> They are not completely overlapping in the single answer case.

This means could be some diversity effect going on

---

In [34]:
individual_groups_answer_per_question = (
    (
        mad_analysis.filter(pl.col("group_constellation").str.len_chars() == 1).pivot(
            values="is_correct",
            index="question_id",
            on="group_constellation",
            aggregate_function="mean",
        )
    )
    .sort("question_id")
    .with_columns(pl.exclude("question_id").cast(bool))
)

df_single_mad = pl.concat(
    [
        individual_groups_answer_per_question.group_by(col)
        .agg(pl.all().exclude(col, "question_id").mean())
        .with_columns(
            pl.when(pl.col(col))
            .then(pl.lit(col + "_correct"))
            .otherwise(pl.lit(col + "_incorrect"))
            .alias("Given"),
            pl.when(pl.col(col)).then(pl.lit(1.0)).otherwise(pl.lit(0.0)).alias(col),
        )
        for col in individual_models_answer_per_question.columns
        if col != "question_id"
    ],
    how="diagonal",
).select("Given", "L", "M", "H")

df_single_mad

Given,L,M,H
str,f64,f64,f64
"""L_incorrect""",0.0,0.386667,0.653333
"""L_correct""",1.0,0.88,0.92
"""M_incorrect""",0.061224,0.0,0.510204
"""M_correct""",0.431373,1.0,0.921569
"""H_incorrect""",0.071429,0.142857,0.0
"""H_correct""",0.319444,0.652778,1.0


In [35]:
combined_ind = individual_models_answer_per_question.join(
    individual_groups_answer_per_question, on="question_id", suffix="_mad"
)
pl.concat(
    [
        combined_ind.group_by(col)
        .agg(pl.all().exclude(col, "question_id").mean())
        .with_columns(
            pl.when(pl.col(col))
            .then(pl.lit(col + "_correct"))
            .otherwise(pl.lit(col + "_incorrect"))
            .alias("Given"),
            pl.when(pl.col(col)).then(pl.lit(1.0)).otherwise(pl.lit(0.0)).alias(col),
        )
        for col in combined_ind.columns
        if col != "question_id"
    ],
    how="diagonal",
).select("Given", pl.exclude("Given"))

Given,L,M,H,H_mad,M_mad,L_mad
str,f64,f64,f64,f64,f64,f64
"""L_correct""",1.0,0.8,0.8,0.828571,0.685714,0.542857
"""L_incorrect""",0.0,0.476923,0.553846,0.661538,0.415385,0.092308
"""M_incorrect""",0.170732,0.0,0.292683,0.463415,0.146341,0.02439
"""M_correct""",0.474576,1.0,0.881356,0.898305,0.762712,0.40678
"""H_correct""",0.4375,0.8125,1.0,0.890625,0.671875,0.34375
…,…,…,…,…,…,…
"""H_mad_correct""",0.402778,0.736111,0.791667,1.0,0.652778,0.319444
"""M_mad_correct""",0.470588,0.882353,0.843137,0.921569,1.0,0.431373
"""M_mad_incorrect""",0.22449,0.285714,0.428571,0.510204,0.0,0.061224


> See Obsidian

#### Is the "nobody is right case" in HHH actually unanimous?

In [36]:
print("Answers where everyone was wrong:")

everyone_wrong = (
    mad_analysis.filter(pl.col("group_constellation") == "HHH")
    .explode(AnalysisColumn.parsed_individual_answers_before.value)
    .with_columns(
        is_individually_correct=comparer(
            pl.col(AnalysisColumn.parsed_individual_answers_before.value),
            pl.col("answer_string"),
        )
    )
    .group_by("question_id")
    .agg(
        number_wrong=pl.len(),
        answers=pl.col(AnalysisColumn.parsed_individual_answers_before.value).implode(),
        is_individually_correct=pl.col("is_individually_correct").implode(),
    )
    .filter(pl.col("is_individually_correct").list.eval(pl.element().not_()).list.all())
    .drop("number_wrong", "is_individually_correct")
)

print(everyone_wrong)
unanimous = (
    everyone_wrong["answers"]
    .filter(everyone_wrong["answers"].list.n_unique() == 1)
    .count()
)
print(f"Of that unanimous: {unanimous}")
print("Contentious:")
print(everyone_wrong["answers"].filter(everyone_wrong["answers"].list.n_unique() != 1))

Answers where everyone was wrong:
shape: (23, 2)
┌─────────────┬─────────────────────────────────┐
│ question_id ┆ answers                         │
│ ---         ┆ ---                             │
│ i64         ┆ list[str]                       │
╞═════════════╪═════════════════════════════════╡
│ 4           ┆ ["D", "D", "D"]                 │
│ 92          ┆ ["C", "D", "C"]                 │
│ 43          ┆ ["___not_parsable___", "___not… │
│ 74          ┆ ["D", "E", "E"]                 │
│ 113         ┆ ["E", "A", "___not_parsable___… │
│ …           ┆ …                               │
│ 6           ┆ ["D", "D", "D"]                 │
│ 13          ┆ ["E", "E", "E"]                 │
│ 109         ┆ ["H", "H", "H"]                 │
│ 3           ┆ ["C", "C", "C"]                 │
│ 30          ┆ ["A", "A", "A"]                 │
└─────────────┴─────────────────────────────────┘
Of that unanimous: 15
Contentious:
shape: (8,)
Series: 'answers' [list[str]]
[
	["C", "D", "C"]
	["D"

C# Intra-Group Correctness Correlation (Aka group diversity)